# THABAT — Thermal Road Screening
### Judge-ready notebook for AOI-A and AOI-B

**Purpose:** reproduce the team's ArcGIS Pro thermal-segment screening from a clean GitHub repository using the supplied sample data.

### Data roles in this notebook
- **AOI-A:** uses the included Landsat surface-temperature raster **and** the AOI-A ArcGIS segment statistics.
- **AOI-B:** uses the existing ArcGIS Pro segment-statistics table. The included Landsat raster is **not** an AOI-B raster and is never used to claim AOI-B spatial coverage.

### What this notebook does
1. Loads the ArcGIS-exported segment-statistics tables for **AOI-A** and **AOI-B**.
2. Loads the small Landsat surface-temperature sample raster used for **AOI-A** thermal context.
3. Converts Landsat Collection 2 Level-2 `ST_B10` digital numbers using the exact team formula:
   **Kelvin = DN × 0.00341802 + 149.0**
   followed by **°C = Kelvin − 273.15**.
4. Treats DN `0` as fill/no-data.
5. Screens existing road segments using their supplied thermal statistics.
6. Reports valid-pixel support so low-support AOI-B results are not presented with the same confidence as AOI-A.
7. Exports clean tables and figures for the repository.

### Interpretation boundary
A thermal anomaly is **not proof of pavement damage**. A single-date thermal product supports spatial screening/context, not a claim of long-term deterioration or causation. Road-level confirmation requires additional observations and physical inspection.

## 1. Repository setup

This notebook is designed to run directly from the THABAT GitHub repository or in Google Colab.

Expected repository structure:

```text
THABAT/
├── notebooks/
│   └── 02_main_analysis.ipynb
├── data/
│   └── sample_input/
│       ├── AOI_A_Segment_LST_Stats_excel.xls
│       ├── aoiB_Segment_stat.xls
│       └── Landsat_ST_AOI_Clip.tif
└── results/
```

**Important:** `Landsat_ST_AOI_Clip.tif` is the thermal sample for **AOI-A only**. AOI-B is analyzed from its ArcGIS Pro segment-statistics table; do not treat the AOI-A raster as AOI-B coverage.

No Google Drive mount, personal path, API key, or secret is required.

In [ ]:
from pathlib import Path
import sys
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Exact public repository shown in the team's GitHub screenshot.
REPO_URL = "https://github.com/DanaAlhammadi/Arab-Youth-Space-Hackathon-2026-813-Challenge.git"
REPO_DIR_NAME = "Arab-Youth-Space-Hackathon-2026-813-Challenge"

def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if running_in_colab():
    # Colab opens the notebook itself under /content, but the repository data
    # are not automatically present. Clone the repository once.
    from pathlib import Path

    repo_dir = Path("/content") / REPO_DIR_NAME

    if not (repo_dir / "data" / "sample_input").exists():
        print("Cloning THABAT repository...")
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(
                "GitHub clone failed.\n"
                f"Repository URL: {REPO_URL}\n"
                f"Git error: {result.stderr.strip()}"
            )
        print("Repository cloned successfully.")
    else:
        print("Repository already available in /content.")
    ROOT = repo_dir
else:
    # Running locally from the repository.
    HERE = Path.cwd()
    if (HERE / "data" / "sample_input").exists():
        ROOT = HERE
    elif (HERE.parent / "data" / "sample_input").exists():
        ROOT = HERE.parent
    else:
        raise FileNotFoundError(
            "Could not find data/sample_input/. "
            "Run this notebook from the THABAT repository."
        )

DATA_DIR = ROOT / "data" / "sample_input"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FILE_A = DATA_DIR / "AOI_A_Segment_LST_Stats_excel.xls"
FILE_B = DATA_DIR / "aoiB_Segment_stat.xls"
RASTER_FILE = DATA_DIR / "Landsat_ST_AOI_Clip.tif"

required_files = [FILE_A, FILE_B, RASTER_FILE]
missing = [p for p in required_files if not p.exists()]

print(f"Repository root: {ROOT}")
print(f"Running in Colab: {running_in_colab()}")
print("\nRequired sample files:")

for p in required_files:
    if p.exists():
        print(f"  [FOUND] {p.relative_to(ROOT)} ({p.stat().st_size / (1024**2):.2f} MB)")
    else:
        print(f"  [MISSING] {p.relative_to(ROOT)}")

if missing:
    raise FileNotFoundError(
        "One or more required sample files are missing. "
        "Upload them to data/sample_input/ in GitHub before running the notebook."
    )

# The .xls inputs require xlrd. Install only when it is missing.
try:
    import xlrd  # noqa: F401
except ImportError:
    print("\nInstalling xlrd for the .xls sample tables...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "xlrd>=2.0.1,<3"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Could not install xlrd: {result.stderr.strip()}")

print("\nEnvironment setup complete. Continue with the next cell.")

## 2. Load the ArcGIS Pro segment statistics

The road segments have already been delineated in ArcGIS Pro, so this notebook **does not segment roads again**.

The Excel tables are the existing segment-level thermal summaries. The notebook reads them as the trusted input tables and analyses them consistently for both AOIs.

In [ ]:
def pick_column(df, keywords):
    # Return the first column whose name contains any keyword.
    cols = [str(c) for c in df.columns]
    for key in keywords:
        for original, label in zip(df.columns, cols):
            if key.lower() in label.lower():
                return original
    return None


def load_segment_table(path, aoi_name):
    if not path.exists():
        raise FileNotFoundError(f"Missing segment table: {path}")

    df = pd.read_excel(path)

    id_col = pick_column(df, ["segment", "segment_id", "id"])
    min_col = pick_column(df, ["min"])
    median_col = pick_column(df, ["median"])
    max_col = pick_column(df, ["max"])
    count_col = pick_column(df, ["count"])

    if id_col is None or median_col is None:
        raise ValueError(
            f"{path.name}: could not identify Segment ID and median-temperature columns. "
            f"Columns found: {list(df.columns)}"
        )

    out = pd.DataFrame({
        "aoi_id": aoi_name,
        "segment_id": df[id_col].astype(str).str.strip(),
        "t_median": pd.to_numeric(df[median_col], errors="coerce"),
    })

    if min_col is not None:
        out["t_min"] = pd.to_numeric(df[min_col], errors="coerce")
    if max_col is not None:
        out["t_max"] = pd.to_numeric(df[max_col], errors="coerce")
    if count_col is not None:
        out["valid_pixel_count"] = pd.to_numeric(df[count_col], errors="coerce")

    if "t_min" not in out.columns or "t_max" not in out.columns:
        raise ValueError(
            f"{path.name}: min/max temperature columns are required for the spread feature."
        )

    out["t_range"] = out["t_max"] - out["t_min"]
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["t_median", "t_min", "t_max"]).reset_index(drop=True)

    return out


A = load_segment_table(FILE_A, "AOI-A")
B = load_segment_table(FILE_B, "AOI-B")

print("AOI-A segments:", A["segment_id"].tolist())
print("AOI-B segments:", B["segment_id"].tolist())

display(A)
display(B)

## 3. Basic data-quality checks

These checks make the notebook safer for judging and catch obvious problems before scoring.

In [ ]:
def quality_report(df):
    report = {
        "segments": int(len(df)),
        "median_min_c": float(df["t_median"].min()),
        "median_max_c": float(df["t_median"].max()),
        "range_min_c": float(df["t_range"].min()),
        "range_max_c": float(df["t_range"].max()),
    }
    if "valid_pixel_count" in df.columns:
        report["min_valid_pixels"] = float(df["valid_pixel_count"].min())
        report["max_valid_pixels"] = float(df["valid_pixel_count"].max())
    return pd.Series(report)

print("AOI-A quality summary")
display(quality_report(A))

print("AOI-B quality summary")
display(quality_report(B))

# Guardrails against obvious bad conversions/fill contamination.
for name, df in [("AOI-A", A), ("AOI-B", B)]:
    if (df["t_median"] < -20).any():
        print(f"WARNING: {name} contains unusually low median temperature values. Check fill/QA handling.")
    if (df["t_median"] > 80).any():
        print(f"WARNING: {name} contains unusually high median temperature values. Check scaling/units.")

### AOI-B support warning

AOI-B currently has only a small number of valid thermal pixels per segment in the supplied ArcGIS statistics (2–3 in the current table). Therefore, any AOI-B screening result should be treated as **low-support / low-confidence thermal context**, not as an equivalent-strength result to AOI-A.

The notebook keeps this limitation visible rather than hiding it.

## 4. Landsat Collection 2 Level-2 `ST_B10` → temperature — AOI-A

For the team's Landsat workflow, the exact conversion is:

**Kelvin = DN × 0.00341802 + 149.0**

then:

**°C = Kelvin − 273.15**

Pixel value **0** is treated as a fill/no-data value before scaling.

> **Important:** this formula is specifically for Landsat Collection 2 Level-2 `ST_B10`. It must not be applied to ECOSTRESS.

> **AOI-A only:** `Landsat_ST_AOI_Clip.tif` is the raster supplied for AOI-A. It is **not** used for AOI-B.

The raster section demonstrates the correct conversion and produces an AOI-A thermal-context map. The segment-level AOI-B results remain based on the separate ArcGIS Pro statistics table.

In [ ]:
try:
    import rasterio
except ImportError as exc:
    raise ImportError(
        "rasterio is required for the Landsat raster section. "
        "Install it with the repository requirements.txt."
    ) from exc

with rasterio.open(RASTER_FILE) as src:
    raw = src.read(1)
    raster_crs = src.crs
    raster_transform = src.transform
    raster_nodata = src.nodata

print("Raster:", RASTER_FILE.name)
print("Shape:", raw.shape)
print("CRS:", raster_crs)
print("Raster nodata metadata:", raster_nodata)
print("Raw dtype:", raw.dtype)
print("Raw min/max:", np.nanmin(raw), np.nanmax(raw))

# Official team conversion for Landsat C2 L2 ST_B10.
dn = raw.astype("float32")
dn[dn == 0] = np.nan

temperature_k = dn * 0.00341802 + 149.0
temperature_c = temperature_k - 273.15

valid = temperature_c[np.isfinite(temperature_c)]
if valid.size == 0:
    raise ValueError("No valid Landsat temperature pixels remain after fill-value masking.")

print("Valid pixels:", valid.size)
print("Temperature percentiles (2, 50, 98 °C):",
      np.percentile(valid, [2, 50, 98]))

if np.nanmin(temperature_c) < -20:
    print("WARNING: unusually cold pixel detected; verify QA/fill handling.")
if np.nanmax(temperature_c) > 80:
    print("WARNING: unusually hot pixel detected; verify QA/fill handling.")

## 5. Landsat thermal context map

This is a **context map**, not a pavement-damage map.

The colour scale is clipped to the 2nd–98th percentiles so that a small number of extreme values do not dominate the view.

In [ ]:
vmin, vmax = np.percentile(valid, [2, 98])

plt.figure(figsize=(10, 6))
plt.imshow(temperature_c, cmap="inferno", vmin=vmin, vmax=vmax)
plt.colorbar(label="Surface temperature (°C)")
plt.title("THABAT — Landsat Surface Temperature Context")
plt.axis("off")
plt.tight_layout()

thermal_map_path = RESULTS_DIR / "landsat_thermal_context.png"
plt.savefig(thermal_map_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved: {thermal_map_path}")

## 6. Transparent segment-level thermal screening

The road segments have already been delineated in ArcGIS Pro, so this notebook does **not** segment the roads again.

The supplied Excel tables contain the segment-level thermal summaries. We calculate a **screening priority**, not a damage probability.

### Method

For each AOI separately:

- `z_median`: robust standardized median temperature
- `t_range`: maximum − minimum temperature within the segment summary
- `z_range`: robust standardized spread
- `valid_pixel_count`: number of valid thermal pixels represented in the supplied segment summary

The screening score is:

`priority_score = max(z_median, 0) + max(z_range, 0)`

This is intentionally simple and interpretable. It should be presented as **thermal screening / inspection-priority evidence**, not proof of pavement deterioration.

### Support flag

Because the current AOI-B table contains only 2–3 valid pixels per segment, the notebook also reports:

- `HIGH SUPPORT` for 30 or more valid pixels
- `MEDIUM SUPPORT` for 10–29
- `LOW SUPPORT` for fewer than 10

These thresholds are **reporting thresholds for transparency**, not a calibrated confidence model.

In [ ]:
def robust_z(series):
    # Robust z-score using median absolute deviation; fallback to std when MAD is zero.
    s = pd.to_numeric(series, errors="coerce").astype(float)
    med = s.median()
    mad = (s - med).abs().median() * 1.4826

    if not np.isfinite(mad) or mad == 0:
        std = s.std(ddof=0)
        if not np.isfinite(std) or std == 0:
            return pd.Series(np.zeros(len(s)), index=s.index)
        return (s - med) / std

    return (s - med) / mad


def score_segments(df):
    out = df.copy()

    out["z_median"] = robust_z(out["t_median"])
    out["z_range"] = robust_z(out["t_range"])

    # Only positive anomalies increase screening priority.
    out["positive_thermal_anomaly"] = out["z_median"].clip(lower=0)
    out["spread_anomaly"] = out["z_range"].clip(lower=0)

    out["priority_score"] = (
        out["positive_thermal_anomaly"] +
        out["spread_anomaly"]
    )

    if "valid_pixel_count" in out.columns:
        out["support_flag"] = np.select(
            [
                out["valid_pixel_count"] >= 30,
                out["valid_pixel_count"] >= 10,
            ],
            [
                "HIGH SUPPORT",
                "MEDIUM SUPPORT",
            ],
            default="LOW SUPPORT"
        )
    else:
        out["support_flag"] = "NOT RECORDED"

    # Keep the ranking interpretable, but do not imply a damage probability.
    out["screening_flag"] = np.select(
        [
            (out["priority_score"] >= 2.5) & (out["support_flag"] == "HIGH SUPPORT"),
            (out["priority_score"] >= 2.5),
            (out["priority_score"] >= 1.0),
        ],
        [
            "HIGHER PRIORITY",
            "HIGHER PRIORITY / LOW SUPPORT",
            "MODERATE",
        ],
        default="LOWER PRIORITY"
    )

    return out.sort_values(
        ["priority_score", "t_median"],
        ascending=[False, False]
    ).reset_index(drop=True)


A_scored = score_segments(A)
B_scored = score_segments(B)

display(A_scored)
display(B_scored)

## 7. Compare the two AOIs

This comparison is for **screening context only**. Scores are calculated separately within each AOI, so the result says which segments are unusual relative to the other segments in the same AOI.

In [ ]:
for name, df in [("AOI-A", A_scored), ("AOI-B", B_scored)]:
    print(f"\n{name}")
    cols = [
        "segment_id", "t_median", "t_range",
        "z_median", "z_range",
        "priority_score", "screening_flag"
    ]
    display(df[cols].reset_index(drop=True))

## 8. Output figures

The figures below are designed to be readable in the final submission and to make the screening logic visible.

The titles and captions use **Thermal Screening** rather than "damage detection". AOI-B support limitations are retained in the exported table and should be stated alongside the figure when presented.

In [ ]:
def make_summary_figure(df, aoi_name, out_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    axes[0].bar(df["segment_id"].astype(str), df["t_median"])
    axes[0].set_title(f"{aoi_name}: Segment Median LST")
    axes[0].set_ylabel("Median surface temperature (°C)")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].scatter(df["t_median"], df["t_range"])
    for _, row in df.iterrows():
        axes[1].annotate(
            str(row["segment_id"]),
            (row["t_median"], row["t_range"]),
            xytext=(4, 4),
            textcoords="offset points"
        )
    axes[1].set_xlabel("Median LST (°C)")
    axes[1].set_ylabel("Internal spread (max − min, °C)")
    axes[1].set_title("Thermal profile by segment")

    plot_df = df.sort_values("priority_score")
    labels = []
    for _, row in plot_df.iterrows():
        sid = str(row["segment_id"])
        labels.append(sid)
    axes[2].barh(labels, plot_df["priority_score"])
    axes[2].set_xlabel("Thermal screening priority score")
    axes[2].set_title("Thermal screening priority")

    fig.suptitle(
        f"THABAT — {aoi_name} Thermal Screening\n"
        "Screening evidence only; not proof of pavement damage",
        y=1.03
    )
    plt.tight_layout()

    path = RESULTS_DIR / out_name
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


make_summary_figure(A_scored, "AOI-A", "AOI_A_thermal_screening_summary.png")
make_summary_figure(B_scored, "AOI-B", "AOI_B_thermal_screening_summary.png")

## 9. Export judge-readable tables

These CSV files are generated from the notebook so the displayed results and downloadable tables remain consistent.

In [ ]:
a_cols = [
    "aoi_id", "segment_id", "t_min", "t_median", "t_max",
    "t_range", "valid_pixel_count", "z_median", "z_range",
    "positive_thermal_anomaly", "spread_anomaly",
    "priority_score", "support_flag", "screening_flag"
]

b_cols = a_cols.copy()

A_out = A_scored[[c for c in a_cols if c in A_scored.columns]]
B_out = B_scored[[c for c in b_cols if c in B_scored.columns]]

A_csv = RESULTS_DIR / "AOI_A_thermal_screening.csv"
B_csv = RESULTS_DIR / "AOI_B_thermal_screening.csv"

A_out.to_csv(A_csv, index=False)
B_out.to_csv(B_csv, index=False)

print("Saved:")
print(A_csv)
print(B_csv)

## 10. Final interpretation

### AOI-A
The supplied Landsat raster provides a thermal-context view for AOI-A, while the ArcGIS Pro segment table provides the segment-level statistics. The current screening identifies thermally unusual segments that may deserve inspection.

### AOI-B
AOI-B is represented by the supplied ArcGIS Pro segment-statistics table. The included Landsat raster is **not** an AOI-B raster. Current AOI-B segment statistics have only 2–3 valid thermal pixels per segment, so their screening results are **low-support** and should not be presented as equivalent-strength evidence to AOI-A.

### What this PoC can support
- identification of thermally unusual segments relative to other segments within an AOI;
- an interpretable inspection-priority ranking;
- a reproducible thermal-context workflow;
- transparent reporting of data-support limitations.

### What this PoC cannot support by itself
- proving pavement damage;
- proving rainfall caused deterioration;
- diagnosing cracks, potholes, rutting, or structural failure;
- claiming that the hottest segment is necessarily the most damaged.

**claim:**

> THABAT uses satellite-derived environmental evidence to screen road segments and prioritize locations for physical inspection. Thermal anomalies provide contextual evidence; they do not independently establish pavement damage or causation.